In [ ]:
#| default_exp core

# Core

> Authentication, compliance profiles, label helpers, project bootstrap, and `GenAIStack`.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
from fastcore.basics import store_attr

from gcpeasy._util import _log, translate_error, update_iam_policy

try:
    import google.auth
    import google.auth.transport.requests
    import google.oauth2.service_account as sa_creds
    from google.cloud import resourcemanager_v3  # noqa: F401
except ImportError:
    pass  # optional at import time; raised at runtime if needed

## Compliance profiles

In [ ]:
#| export
HIPAA = dict(
    encryption=True,
    tls_min='1.2',
    audit=True,
    audit_all_services=True,  # HIPAA requires Data Access logs on every service
    multi_region=True,
    backup_retention=35,
    deletion_protection=True,
    labels={'compliance': 'hipaa'},
)

ISO27001 = dict(
    encryption=True,
    audit=True,
    managed_sa=True,
    least_privilege=True,
    tls_min='1.2',
    labels={'compliance': 'iso27001'},
)

SOC2 = dict(
    encryption=True,
    audit=True,
    mfa_required=True,
    backup_retention=7,
    labels={'compliance': 'soc2'},
)

## APIs commonly required by gcpeasy operations

In [ ]:
#| export
#: Minimal set of APIs needed for VM + Cloud Run + Artifact Registry deploys.
REQUIRED_APIS = [
    'compute.googleapis.com',
    'iam.googleapis.com',
    'iamcredentials.googleapis.com',
    'cloudresourcemanager.googleapis.com',
    'serviceusage.googleapis.com',
    'storage.googleapis.com',
    'secretmanager.googleapis.com',
    'artifactregistry.googleapis.com',
    'run.googleapis.com',
    'cloudbuild.googleapis.com',
    'logging.googleapis.com',
]

#: Additional APIs for the full GenAI stack (Vertex, Firestore, Memorystore, …).
GENAI_APIS = REQUIRED_APIS + [
    'aiplatform.googleapis.com',
    'firestore.googleapis.com',
    'redis.googleapis.com',
    'sqladmin.googleapis.com',
    'discoveryengine.googleapis.com',
    'iap.googleapis.com',
    'dns.googleapis.com',
]


class GCPAuth:
    """Application Default Credentials wrapper for GCP. Reads ``GOOGLE_CLOUD_PROJECT``
    and ``GOOGLE_CLOUD_REGION`` from env. Pass ``service_account_file=`` for key-based auth."""

    def __init__(self, project=None, region=None, service_account_file=None,
                 impersonate_sa=None):
        store_attr()
        self.project = project or os.environ.get('GOOGLE_CLOUD_PROJECT') or os.environ.get('GCLOUD_PROJECT')
        self.region  = region  or os.environ.get('GOOGLE_CLOUD_REGION', 'us-central1')
        if not self.project:
            raise ValueError('project required; set GOOGLE_CLOUD_PROJECT or pass project=')

        if service_account_file:
            self.credentials = sa_creds.Credentials.from_service_account_file(
                service_account_file,
                scopes=['https://www.googleapis.com/auth/cloud-platform'],
            )
        else:
            self.credentials, _ = google.auth.default(
                scopes=['https://www.googleapis.com/auth/cloud-platform']
            )

        if impersonate_sa:
            from google.auth import impersonated_credentials
            self.credentials = impersonated_credentials.Credentials(
                source_credentials=self.credentials,
                target_principal=impersonate_sa,
                target_scopes=['https://www.googleapis.com/auth/cloud-platform'],
            )

    def __repr__(self):
        return f'GCPAuth(project={self.project!r}, region={self.region!r})'

## Asset Inventory: label search

In [ ]:
#| export
def label_resources(auth, labels: dict) -> list:
    """List GCP project resources matching ``labels`` using Cloud Asset Inventory.

    When ``labels`` is empty, all labeled resources in the project scope are
    returned (the previous implementation built an empty ``query`` string,
    which the API rejects).
    """
    from google.cloud import asset_v1
    client = asset_v1.AssetServiceClient(credentials=auth.credentials)
    kwargs = dict(scope=f'projects/{auth.project}')
    if labels:
        kwargs['query'] = ' AND '.join(f'labels.{k}={v}' for k, v in labels.items())
    req = asset_v1.SearchAllResourcesRequest(**kwargs)
    return list(client.search_all_resources(request=req))


def list_labeled_resources(auth) -> list:
    """List all resources in the project (no label filter)."""
    return label_resources(auth, {})

## Audit logging

In [ ]:
#| export
#: Sensible default services to enable Data Access audit logs on.  Avoids the
#: very large log volume (and bill) that ``allServices`` produces for noisy
#: services like ``compute.googleapis.com``.
DEFAULT_AUDIT_SERVICES = (
    'aiplatform.googleapis.com',
    'storage.googleapis.com',
    'secretmanager.googleapis.com',
    'bigquery.googleapis.com',
)


def enable_data_access_audit(auth, service: str = None,
                             services: list = None,
                             all_services: bool = False) -> dict:
    """Enable Data Access audit logs (ADMIN_READ, DATA_READ, DATA_WRITE).

    The previous default was ``service='allServices'`` which produces enormous
    Cloud Logging bills on real workloads.  We now default to a curated set of
    GenAI-relevant services (``DEFAULT_AUDIT_SERVICES``) and require an
    explicit ``all_services=True`` (or HIPAA ``audit_all_services``) for the
    legacy behavior.

    Idempotent: services that already have all three log types are skipped.
    Uses an etag-aware read-modify-write to avoid clobbering concurrent
    binding changes.
    """
    import googleapiclient.discovery
    crm = googleapiclient.discovery.build(
        'cloudresourcemanager', 'v1', credentials=auth.credentials
    )

    if all_services:
        targets = ['allServices']
    elif service is not None:
        targets = [service]
    elif services:
        targets = list(services)
    else:
        targets = list(DEFAULT_AUDIT_SERVICES)

    statuses = {}

    def _mutate(policy):
        changed = False
        audit_configs = policy.setdefault('auditConfigs', [])
        existing_by_svc = {c.get('service'): c for c in audit_configs}
        wanted = {'ADMIN_READ', 'DATA_READ', 'DATA_WRITE'}
        for svc in targets:
            cfg = existing_by_svc.get(svc)
            if cfg:
                have = {c.get('logType') for c in cfg.get('auditLogConfigs', [])}
                if wanted.issubset(have):
                    statuses[svc] = 'already_configured'
                    continue
                cfg['auditLogConfigs'] = [{'logType': t} for t in sorted(wanted)]
                statuses[svc] = 'updated'
                changed = True
            else:
                audit_configs.append({
                    'service': svc,
                    'auditLogConfigs': [{'logType': t} for t in sorted(wanted)],
                })
                statuses[svc] = 'enabled'
                changed = True
        return changed

    update_iam_policy(crm, auth.project, _mutate)
    return {'project': auth.project, 'services': statuses}

## Project bootstrap (B6) and preflight (B7)

In [ ]:
#| export
def enable_apis(auth, apis: list = None, wait: bool = True) -> dict:
    """Enable Google Cloud APIs on the project (idempotent).

    Defaults to :data:`REQUIRED_APIS`.  Returns a mapping of api → status
    (``'already_enabled'`` or ``'enabled'``).
    """
    import googleapiclient.discovery
    apis = list(apis) if apis else list(REQUIRED_APIS)
    su = googleapiclient.discovery.build(
        'serviceusage', 'v1', credentials=auth.credentials
    )
    parent = f'projects/{auth.project}'
    # Identify already-enabled services to avoid no-op enable calls.
    enabled = set()
    req = su.services().list(parent=parent, filter='state:ENABLED', pageSize=200)
    while req is not None:
        page = req.execute()
        for s in page.get('services', []):
            enabled.add(s['config']['name'])
        req = su.services().list_next(previous_request=req, previous_response=page)
    statuses = {}
    to_enable = [a for a in apis if a not in enabled]
    for a in apis:
        if a in enabled:
            statuses[a] = 'already_enabled'
    if to_enable:
        _log(f'enable_apis: enabling {len(to_enable)} services: {", ".join(to_enable)}')
        op = su.services().batchEnable(
            parent=parent,
            body={'serviceIds': to_enable},
        ).execute()
        if wait:
            _wait_su(su, op)
        for a in to_enable:
            statuses[a] = 'enabled'
    return statuses


def _wait_su(su, op, timeout: int = 300):
    import time as _t
    name = op.get('name')
    if not name:
        return op
    start = _t.monotonic()
    while True:
        done = su.operations().get(name=name).execute()
        if done.get('done'):
            if 'error' in done:
                raise RuntimeError(f'enable_apis failed: {done["error"]}')
            return done
        if _t.monotonic() - start > timeout:
            raise TimeoutError(f'enable_apis timed out after {timeout}s')
        _t.sleep(2)


def preflight(auth, apis: list = None, require_billing: bool = True) -> dict:
    """Run a fast pre-deploy check against the project. Raises ``GcpEasyError``
    on failure with concrete remediation; returns a status dict on success.

    Verifies:
      * ADC credentials resolve and ``project`` is set
      * Billing is enabled on the project (when ``require_billing``)
      * The given ``apis`` (default :data:`REQUIRED_APIS`) are enabled
    """
    import googleapiclient.discovery
    from ._util import GcpEasyError
    out = {'project': auth.project, 'region': auth.region}

    # ADC sanity
    if not getattr(auth, 'credentials', None):
        raise GcpEasyError('ADC not initialised; run `gcloud auth application-default login`.')

    # Billing
    if require_billing:
        try:
            billing = googleapiclient.discovery.build(
                'cloudbilling', 'v1', credentials=auth.credentials
            )
            info = billing.projects().getBillingInfo(
                name=f'projects/{auth.project}'
            ).execute()
            if not info.get('billingEnabled'):
                raise GcpEasyError(
                    f'Billing is not enabled on project {auth.project!r}; '
                    f'enable it at https://console.cloud.google.com/billing/linkedaccount?project={auth.project}'
                )
            out['billing'] = info.get('billingAccountName', 'enabled')
        except GcpEasyError:
            raise
        except Exception as e:  # noqa: BLE001
            # Billing API may itself be disabled; surface a translated error.
            raise translate_error(e, 'preflight billing check')

    # APIs
    apis = list(apis) if apis else list(REQUIRED_APIS)
    su = googleapiclient.discovery.build(
        'serviceusage', 'v1', credentials=auth.credentials
    )
    parent = f'projects/{auth.project}'
    enabled = set()
    req = su.services().list(parent=parent, filter='state:ENABLED', pageSize=200)
    while req is not None:
        page = req.execute()
        for s in page.get('services', []):
            enabled.add(s['config']['name'])
        req = su.services().list_next(previous_request=req, previous_response=page)
    missing = [a for a in apis if a not in enabled]
    out['apis_enabled'] = sorted(set(apis) & enabled)
    out['apis_missing'] = missing
    if missing:
        raise GcpEasyError(
            'Required APIs not enabled: ' + ', '.join(missing) +
            '\n  -> Run gcpeasy.core.enable_apis(auth) or `gcpeasy enable-apis`.'
        )
    return out

## GenAIStack

In [ ]:
#| export
class GenAIStack:
    """Provision a full enterprise GenAI stack on GCP in one call."""

    def __init__(self, auth: GCPAuth, name: str, compliance: dict = None):
        store_attr()
        self._resources = {}
        self._compliance = compliance or {}

    def enable_required_apis(self, extra: list = None):
        """Enable the APIs needed for the GenAI stack on this project."""
        apis = list(GENAI_APIS) + list(extra or [])
        return enable_apis(self.auth, apis)

    def provision(
        self,
        vertex_ai: bool = True,
        vector_search: bool = False,
        gcs: bool = True,
        firestore: bool = True,
        cloud_sql: bool = False,
        memorystore: bool = True,
        gke: bool = False,
        secret_manager: bool = True,
        enable_apis_first: bool = True,
    ) -> dict:
        """Provision selected GCP services. Returns dict of resource identifiers."""
        from gcpeasy.network import create_service_account, bind_iam_role, create_secret
        from gcpeasy.data import create_bucket, create_collection, create_redis, create_postgres
        from gcpeasy.ai import create_vector_search_index

        if enable_apis_first:
            try:
                self.enable_required_apis()
            except Exception as e:  # noqa: BLE001 - non-fatal
                _log(f'GenAIStack: enable_apis skipped ({e})')

        name = self.name
        co = self._compliance
        labels = dict(co.get('labels', {}))
        labels.update({'gcpeasy': name})

        # Enable audit logging when compliance profile requests it
        if co.get('audit'):
            enable_data_access_audit(
                self.auth,
                all_services=bool(co.get('audit_all_services')),
            )

        # Service Account
        sa = create_service_account(self.auth, f'{name}-sa',
                                    display_name=f'{name} GenAI SA')
        bind_iam_role(self.auth, sa['email'], 'roles/aiplatform.user')
        self._resources['service_account'] = sa['email']

        if gcs:
            bucket = create_bucket(self.auth, f'{name}-data', labels=labels, **co)
            self._resources['gcs_bucket'] = bucket['name']

        if firestore:
            coll = create_collection(self.auth, name)
            self._resources['firestore_collection'] = coll

        if memorystore:
            redis = create_redis(self.auth, f'{name}-cache', labels=labels, **co)
            self._resources['memorystore'] = redis.get('name')

        if cloud_sql:
            pg = create_postgres(self.auth, f'{name}-db', labels=labels, **co)
            self._resources['cloud_sql'] = pg.get('name')

        if vector_search:
            idx = create_vector_search_index(self.auth, f'{name}-index',
                                             dimensions=768, wait=False)
            self._resources['vector_search_index'] = idx.get('name')

        if secret_manager:
            sec = create_secret(self.auth, f'{name}/api-key', 'placeholder',
                                labels=labels)
            self._resources['secret'] = sec.get('name')

        return self._resources

    def destroy(self) -> dict:
        """Delete all resources tracked in :pyattr:`_resources` (best-effort)."""
        from gcpeasy.network import delete_secret, delete_service_account
        from gcpeasy.data import delete_bucket, delete_redis, delete_postgres
        results = {}
        order = [
            ('secret', delete_secret),
            ('memorystore', lambda auth, name: delete_redis(auth, _short(name))),
            ('cloud_sql', lambda auth, name: delete_postgres(auth, _short(name))),
            ('gcs_bucket', delete_bucket),
            ('service_account', lambda auth, email: delete_service_account(auth, email)),
        ]
        for key, fn in order:
            val = self._resources.get(key)
            if not val:
                continue
            try:
                fn(self.auth, val)
                results[key] = 'deleted'
            except Exception as e:  # noqa: BLE001
                results[key] = f'error: {e}'
        return results

    def summary(self) -> dict:
        "Return provisioned resource identifiers."
        return self._resources


def _short(resource_path: str) -> str:
    """Return the leaf segment of a fully-qualified GCP resource name."""
    return resource_path.rsplit('/', 1)[-1] if '/' in resource_path else resource_path

### Tests — compliance profiles

In [ ]:
#| hide
from fastcore.test import test_eq
for p in (HIPAA, ISO27001, SOC2):
    test_eq(p['encryption'], True)
    assert 'labels' in p
test_eq(HIPAA['audit_all_services'], True)

### Tests — `REQUIRED_APIS` ⊆ `GENAI_APIS`

In [ ]:
#| hide
assert set(REQUIRED_APIS).issubset(set(GENAI_APIS))

### Tests — default audit services exclude `allServices`

In [ ]:
#| hide
assert 'allServices' not in DEFAULT_AUDIT_SERVICES

### Tests — `GenAIStack` exposes `provision`/`summary`/`destroy`

In [ ]:
#| hide
for n in ('provision', 'summary', 'destroy'):
    assert hasattr(GenAIStack, n), n

### Tests — ported from `tests/test_core.py`


In [ ]:
#| hide
import sys as _sys
from unittest.mock import MagicMock, patch

# When nbdev-test runs the notebook, exported source is inlined into the
# kernel's globals (``__main__``).  To intercept names looked up by inlined
# functions (e.g. ``update_iam_policy``), patch attributes on ``__main__``,
# not on ``gcpeasy.core``.
_kernel = _sys.modules['__main__']

class _MockAuth:
    project = 'test-project'
    region = 'us-central1'
    credentials = MagicMock(name='credentials')

def _mock_auth():
    return _MockAuth()


In [ ]:
#| hide
# Compliance profiles — required keys
for p in (HIPAA, ISO27001, SOC2):
    assert p['encryption'] is True
    assert 'labels' in p
assert HIPAA['audit_all_services'] is True
assert ISO27001.get('audit_all_services') is None  # services list only


In [ ]:
#| hide
# REQUIRED_APIS ⊆ GENAI_APIS
assert set(REQUIRED_APIS).issubset(set(GENAI_APIS))


In [ ]:
#| hide
# Default audit logging targets a curated services list, NOT 'allServices'
captured = {}
def fake_update(crm, resource, mutate):
    policy = {'bindings': [], 'etag': 'e', 'auditConfigs': []}
    mutate(policy)
    captured['policy'] = policy
    return policy

with patch.object(_kernel, 'update_iam_policy', side_effect=fake_update), \
     patch('googleapiclient.discovery.build', return_value=MagicMock()):
    result = enable_data_access_audit(_mock_auth())

services = {c['service'] for c in captured['policy']['auditConfigs']}
assert 'allServices' not in services
assert services == set(DEFAULT_AUDIT_SERVICES)
for cfg in captured['policy']['auditConfigs']:
    types_ = {t['logType'] for t in cfg['auditLogConfigs']}
    assert types_ == {'ADMIN_READ', 'DATA_READ', 'DATA_WRITE'}
assert all(s in result['services'] for s in DEFAULT_AUDIT_SERVICES)


In [ ]:
#| hide
# all_services=True is honoured explicitly
captured = {}
def fake_update(crm, resource, mutate):
    policy = {'bindings': [], 'etag': 'e', 'auditConfigs': []}
    mutate(policy); captured['policy'] = policy; return policy
with patch.object(_kernel, 'update_iam_policy', side_effect=fake_update), \
     patch('googleapiclient.discovery.build', return_value=MagicMock()):
    enable_data_access_audit(_mock_auth(), all_services=True)
services = {c['service'] for c in captured['policy']['auditConfigs']}
assert services == {'allServices'}


In [ ]:
#| hide
# Idempotent when all 3 log types are already configured for every default service
starting = {'bindings': [], 'etag': 'e', 'auditConfigs': [
    {'service': s, 'auditLogConfigs': [
        {'logType': 'ADMIN_READ'},
        {'logType': 'DATA_READ'},
        {'logType': 'DATA_WRITE'},
    ]}
    for s in DEFAULT_AUDIT_SERVICES
]}
state = {'wrote': False}
def fake_update(crm, resource, mutate):
    policy = dict(starting); policy['auditConfigs'] = list(starting['auditConfigs'])
    if mutate(policy):
        state['wrote'] = True
    return policy
with patch.object(_kernel, 'update_iam_policy', side_effect=fake_update), \
     patch('googleapiclient.discovery.build', return_value=MagicMock()):
    result = enable_data_access_audit(_mock_auth())
assert state['wrote'] is False
for s in DEFAULT_AUDIT_SERVICES:
    assert result['services'][s] == 'already_configured'


In [ ]:
#| hide
# Regression: list_labeled_resources must NOT pass an empty `query` (API rejects '')
fake_client = MagicMock()
fake_client.search_all_resources.return_value = []
captured = {}
class _FakeRequest:
    def __init__(self, **kw): captured.update(kw)
fake_module = MagicMock()
fake_module.AssetServiceClient = MagicMock(return_value=fake_client)
fake_module.SearchAllResourcesRequest = _FakeRequest

a = _mock_auth()
with patch.dict('sys.modules', {'google.cloud.asset_v1': fake_module}), \
     patch('google.cloud.asset_v1', fake_module, create=True):
    list_labeled_resources(a)
assert 'query' not in captured
assert captured.get('scope') == f'projects/{a.project}'
